In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Estilo
sns.set_theme(style="whitegrid", palette="Blues_r")
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11

# Carga
morosidad = pd.read_csv(r'C:\Users\sebga\OneDrive\Documentos\Proyectos\Analisis-SBS\data\processed\morosidad_consolidada.csv')
creditos_tipo = pd.read_csv(r'C:\Users\sebga\OneDrive\Documentos\Proyectos\Analisis-SBS\data\processed\creditos_tipo_consolidado.csv')
creditos_sector = pd.read_csv(r'C:\Users\sebga\OneDrive\Documentos\Proyectos\Analisis-SBS\data\processed\creditos_sector_consolidado.csv')

print(f'Morosidad:        {len(morosidad):,} registros')
print(f'Créditos por tipo: {len(creditos_tipo):,} registros')
print(f'Créditos por sector: {len(creditos_sector):,} registros')

Morosidad:        2,737 registros
Créditos por tipo: 6,018 registros
Créditos por sector: 1,776 registros


In [2]:
BANCOS_MAP = {
    "B. BBVA Perú": "BBVA",
    "B. de Comercio": "Comercio",
    "B. de Crédito del Perú (con sucursales en el exterior)": "BCP",
    "B. Pichincha": "Pichincha",
    "B. Interamericano de Finanzas": "BanBif",
    "Scotiabank Perú": "Scotiabank",
    "B. Falabella Perú": "Falabella",
    "B. Ripley": "Ripley",
    "B. GNB": "GNB",
    "B. Santander Perú": "Santander",
    "B. Azteca Perú": "Azteca",
    "B. ICBC": "ICBC",
    "B. BCI Perú": "BCI",
    "Banco BBVA Perú": "BBVA",
    "Santander Consumer Bank": "Santander",
    "BANCOM": "Compartamos",
    "Compartamos Banco": "Compartamos",

}

for df in [morosidad, creditos_tipo, creditos_sector]:
    df['banco'] = df['banco'].replace(BANCOS_MAP)

print('Bancos en morosidad:', sorted(morosidad['banco'].unique()))

Bancos en morosidad: ['Alfin Banco', 'Azteca', 'BBVA', 'BCI', 'BCP', 'BanBif', 'Bank of China', 'Citibank', 'Comercio', 'Compartamos', 'Falabella', 'GNB', 'ICBC', 'Interbank', 'Mibanco', 'Pichincha', 'Ripley', 'Santander', 'Scotiabank']


In [8]:
# Los valores ya están en escala correcta (%)
# Solo filtrar outliers extremos
morosidad = morosidad[morosidad['morosidad_pct'] <= 30]
print(f"Registros después de limpieza: {len(morosidad)}")

print("\nMorosidad por banco - Diciembre 2024:")
print(
    morosidad[morosidad['periodo'] == '2024-12']
    .groupby('banco')['morosidad_pct']
    .median()
    .sort_values(ascending=False)
    .round(2)
    .to_string()
)

Registros después de limpieza: 923

Morosidad por banco - Diciembre 2024:
banco
BBVA             0.0
Alfin Banco      0.0
BCI              0.0
BCP              0.0
BanBif           0.0
Bank of China    0.0
Citibank         0.0
Compartamos      0.0
Falabella        0.0
GNB              0.0
ICBC             0.0
Interbank        0.0
Mibanco          0.0
Pichincha        0.0
Santander        0.0
Scotiabank       0.0


In [3]:
print("Valores originales del CSV:")
print(morosidad['morosidad_pct'].describe())
print("\nPrimeras filas:")
print(morosidad[['banco', 'tipo_credito', 'morosidad_pct']].head(10))

Valores originales del CSV:
count     2737.000000
mean      1237.406116
std       2460.642751
min          0.000000
25%          0.000000
50%        263.771400
75%        991.874400
max      10000.000000
Name: morosidad_pct, dtype: float64

Primeras filas:
        banco           tipo_credito  morosidad_pct
0        BBVA  Créditos corporativos         5.7047
1    Comercio  Créditos corporativos         0.0413
2         BCP  Créditos corporativos       100.1058
3   Pichincha  Créditos corporativos         0.0049
4      BanBif  Créditos corporativos         2.4922
5  Scotiabank  Créditos corporativos       139.3358
6    Citibank  Créditos corporativos         0.0000
7   Interbank  Créditos corporativos         0.0000
8         GNB  Créditos corporativos         0.0000
9   Santander  Créditos corporativos         0.0080


In [4]:
print("=== MOROSIDAD ===")
print(morosidad.describe())
print("\n=== CRÉDITOS POR TIPO ===")
print(creditos_tipo.describe())
print("\n=== CRÉDITOS POR SECTOR ===")
print(creditos_sector.describe())

=== MOROSIDAD ===
       morosidad_pct
count    2737.000000
mean     1237.406116
std      2460.642751
min         0.000000
25%         0.000000
50%       263.771400
75%       991.874400
max     10000.000000

=== CRÉDITOS POR TIPO ===
       creditos_miles
count    6.018000e+03
mean     7.500670e+05
std      2.855059e+06
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      8.322667e+04
max      4.150499e+07

=== CRÉDITOS POR SECTOR ===
       creditos_miles
count    1.776000e+03
mean     7.515144e+05
std      2.225149e+06
min      0.000000e+00
25%      0.000000e+00
50%      3.501124e+04
75%      4.318605e+05
max      2.061890e+07


In [1]:
# Filtrar solo tipos de crédito principales
tipos_principales = [
    'Créditos de consumo', 'Créditos hipotecarios para vivienda',
    'Créditos a medianas empresas', 'Créditos a grandes empresas',
    'Créditos corporativos', 'Créditos a microempresas'
]

mora_sistema = (
    morosidad[morosidad['tipo_credito'].isin(tipos_principales)]
    .groupby(['periodo', 'tipo_credito'])['morosidad_pct']
    .mean()
    .reset_index()
)

fig, ax = plt.subplots(figsize=(14, 6))
for tipo in tipos_principales:
    data = mora_sistema[mora_sistema['tipo_credito'] == tipo]
    ax.plot(data['periodo'], data['morosidad_pct'], marker='o', linewidth=2, label=tipo)

ax.set_title('Evolución de Morosidad Promedio del Sistema Financiero Peruano\n2020 - 2025', fontsize=14, fontweight='bold')
ax.set_xlabel('Período')
ax.set_ylabel('Morosidad (%)')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('screenshots/morosidad_evolucion.png', dpi=150, bbox_inches='tight')
plt.show()
print("✔ Gráfico guardado")

NameError: name 'morosidad' is not defined

In [ ]:
tipos_consumo = ['Créditos de consumo', 'Créditos de consumo*']

mora_bancos = (
    morosidad[
        morosidad['tipo_credito'].isin(tipos_consumo) &
        morosidad['periodo'].isin(['2020-12', '2024-12'])
    ]
    .groupby(['periodo', 'banco'])['morosidad_pct']
    .mean()
    .reset_index()
)

mora_pivot = mora_bancos.pivot(index='banco', columns='periodo', values='morosidad_pct').dropna()

fig, ax = plt.subplots(figsize=(14, 6))
x = range(len(mora_pivot))
width = 0.35
bars1 = ax.bar([i - width/2 for i in x], mora_pivot['2020-12'], width, label='Dic 2020', color='#2196F3', alpha=0.8)
bars2 = ax.bar([i + width/2 for i in x], mora_pivot['2024-12'], width, label='Dic 2024', color='#F44336', alpha=0.8)

ax.set_title('Morosidad en Créditos de Consumo por Banco\nDiciembre 2020 vs Diciembre 2024', fontsize=14, fontweight='bold')
ax.set_xlabel('Banco')
ax.set_ylabel('Morosidad (%)')
ax.set_xticks(list(x))
ax.set_xticklabels(mora_pivot.index, rotation=45, ha='right')
ax.legend()
plt.tight_layout()
plt.savefig('screenshots/morosidad_bancos_comparativo.png', dpi=150, bbox_inches='tight')
plt.show()
print("✔ Gráfico guardado")

In [ ]:
mora_2024 = (
    morosidad[
        morosidad['tipo_credito'].isin(tipos_consumo) &
        (morosidad['periodo'] == '2024-12')
    ]
    .groupby('banco')['morosidad_pct']
    .mean()
    .sort_values(ascending=False)
    .head(5)
    .reset_index()
)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(mora_2024['banco'], mora_2024['morosidad_pct'], color='#F44336', alpha=0.85)
ax.bar_label(bars, fmt='%.2f%%', padding=3)
ax.set_title('Top 5 Bancos con Mayor Morosidad en Créditos de Consumo\nDiciembre 2024', fontsize=14, fontweight='bold')
ax.set_xlabel('Morosidad (%)')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('screenshots/top5_morosidad_consumo.png', dpi=150, bbox_inches='tight')
plt.show()
print("✔ Gráfico guardado")

In [ ]:
# Bancos principales para no saturar el gráfico
bancos_principales = ['BCP', 'BBVA', 'Scotiabank', 'Interbank', 'BanBif', 'Mibanco']

creditos_evol = (
    creditos_tipo[creditos_tipo['banco'].isin(bancos_principales)]
    .groupby(['periodo', 'banco'])['creditos_miles']
    .sum()
    .reset_index()
)
creditos_evol['creditos_millones'] = creditos_evol['creditos_miles'] / 1000

fig, ax = plt.subplots(figsize=(14, 6))
for banco in bancos_principales:
    data = creditos_evol[creditos_evol['banco'] == banco]
    ax.plot(data['periodo'], data['creditos_millones'], marker='o', linewidth=2.5, label=banco)

ax.set_title('Evolución de Créditos Totales por Banco Principal\n2020 - 2025 (En millones de soles)', fontsize=14, fontweight='bold')
ax.set_xlabel('Período')
ax.set_ylabel('Créditos (Millones S/)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'S/ {x:,.0f}M'))
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('screenshots/creditos_evolucion_bancos.png', dpi=150, bbox_inches='tight')
plt.show()
print("✔ Gráfico guardado")

In [ ]:
market_share = (
    creditos_tipo[creditos_tipo['periodo'] == '2024-12']
    .groupby('banco')['creditos_miles']
    .sum()
    .sort_values(ascending=False)
)

# Agrupar bancos pequeños en "Otros"
top_bancos = market_share.head(6)
otros = pd.Series({'Otros': market_share.iloc[6:].sum()})
market_share_final = pd.concat([top_bancos, otros])

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#1565C0', '#1976D2', '#1E88E5', '#2196F3', '#42A5F5', '#90CAF9', '#BBDEFB']
wedges, texts, autotexts = ax.pie(
    market_share_final.values,
    labels=market_share_final.index,
    autopct='%1.1f%%',
    colors=colors,
    startangle=140,
    pctdistance=0.82
)
for text in autotexts:
    text.set_fontsize(10)
ax.set_title('Participación de Mercado en Créditos\nBanca Múltiple Perú - Diciembre 2024', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('screenshots/market_share_2024.png', dpi=150, bbox_inches='tight')
plt.show()
print("✔ Gráfico guardado")

In [ ]:
top_sectores = (
    creditos_sector[creditos_sector['periodo'] == '2024-12']
    .groupby('sector')['creditos_miles']
    .sum()
    .sort_values(ascending=True)
    .tail(10)
    .reset_index()
)
top_sectores['creditos_millones'] = top_sectores['creditos_miles'] / 1000

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(top_sectores['sector'], top_sectores['creditos_millones'], color='#1976D2', alpha=0.85)
ax.bar_label(bars, fmt='S/ {:,.0f}M', padding=3, fontsize=9)
ax.set_title('Top 10 Sectores Económicos por Volumen de Créditos\nDiciembre 2024 (En millones de soles)', fontsize=14, fontweight='bold')
ax.set_xlabel('Créditos (Millones S/)')
plt.tight_layout()
plt.savefig('screenshots/top10_sectores_2024.png', dpi=150, bbox_inches='tight')
plt.show()
print("✔ Gráfico guardado")

In [ ]:
sectores_clave = ['Comercio', 'Industria Manufacturera', 'Construcción', 'Agricultura, Ganadería, Caza y Silvicultura']

sector_evol = (
    creditos_sector[creditos_sector['sector'].isin(sectores_clave)]
    .groupby(['periodo', 'sector'])['creditos_miles']
    .sum()
    .reset_index()
)
sector_evol['creditos_millones'] = sector_evol['creditos_miles'] / 1000

fig, ax = plt.subplots(figsize=(14, 6))
colors_sector = ['#1565C0', '#F57C00', '#2E7D32', '#6A1B9A']
for sector, color in zip(sectores_clave, colors_sector):
    data = sector_evol[sector_evol['sector'] == sector]
    ax.plot(data['periodo'], data['creditos_millones'], marker='o', linewidth=2.5, label=sector, color=color)

ax.set_title('Evolución de Créditos en Sectores Económicos Clave\n2020 - 2025 (En millones de soles)', fontsize=14, fontweight='bold')
ax.set_xlabel('Período')
ax.set_ylabel('Créditos (Millones S/)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'S/ {x:,.0f}M'))
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('screenshots/sectores_clave_evolucion.png', dpi=150, bbox_inches='tight')
plt.show()
print("✔ Gráfico guardado")

In [ ]:
# Calcular métricas reales para los insights


mora_sistema_2020 = morosidad[morosidad['periodo'] == '2020-12']['morosidad_pct'].mean()
mora_sistema_2024 = morosidad[morosidad['periodo'] == '2024-12']['morosidad_pct'].mean()
mora_sistema_2025 = morosidad[morosidad['periodo'] == '2025-12']['morosidad_pct'].mean()

creditos_2020 = creditos_tipo[creditos_tipo['periodo'] == '2020-12']['creditos_miles'].sum() / 1_000_000
creditos_2024 = creditos_tipo[creditos_tipo['periodo'] == '2024-12']['creditos_miles'].sum() / 1_000_000
crecimiento = ((creditos_2024 - creditos_2020) / creditos_2020) * 100

banco_mayor_mora = morosidad[morosidad['periodo'] == '2024-12'].groupby('banco')['morosidad_pct'].mean().idxmax()
sector_lider = creditos_sector[creditos_sector['periodo'] == '2024-12'].groupby('sector')['creditos_miles'].sum().idxmax()

print("=" * 60)
print("  HALLAZGOS PRINCIPALES")
print("=" * 60)
print(f"  Morosidad promedio dic 2020 : {mora_sistema_2020:.2f}%")
print(f"  Morosidad promedio dic 2024 : {mora_sistema_2024:.2f}%")
print(f"  Morosidad promedio dic 2025 : {mora_sistema_2025:.2f}%")
print(f"  Variación morosidad 2020-2024: {mora_sistema_2024 - mora_sistema_2020:+.2f}pp")
print()
print(f"  Créditos totales dic 2020   : S/ {creditos_2020:.1f}B")
print(f"  Créditos totales dic 2024   : S/ {creditos_2024:.1f}B")
print(f"  Crecimiento cartera 2020-2024: {crecimiento:+.1f}%")
print()
print(f"  Banco mayor morosidad 2024  : {banco_mayor_mora}")
print(f"  Sector con mayor créditos   : {sector_lider}")
print("=" * 60)

In [ ]:
from IPython.display import Markdown

conclusiones = f"""
## Conclusiones del Análisis — Sistema Financiero Peruano (2020-2025)

### Contexto
Análisis de datos públicos de la Superintendencia de Banca, Seguros y AFP (SBS) 
sobre la Banca Múltiple peruana, abarcando {morosidad['banco'].nunique()} instituciones 
financieras en el período 2020-2025.

### Hallazgo 1 — Deterioro sostenido de la calidad crediticia
La morosidad promedio del sistema pasó de **{mora_sistema_2020:.2f}%** en diciembre 2020 
a **{mora_sistema_2024:.2f}%** en diciembre 2024, una variación de 
**{mora_sistema_2024 - mora_sistema_2020:+.2f} puntos porcentuales**. 
Este deterioro es consistente con el fin de las medidas de alivio crediticio 
implementadas durante la pandemia COVID-19 (Decreto de Urgencia 026-2020).

### Hallazgo 2 — Crecimiento sostenido de la cartera
A pesar del aumento en morosidad, la cartera total de créditos creció 
**{crecimiento:+.1f}%** entre 2020 y 2024, reflejando la recuperación económica 
post-pandemia y la expansión del crédito de consumo e hipotecario.

### Hallazgo 3 — Concentración del mercado
El mercado de créditos mantiene una alta concentración. BCP, BBVA e Interbank 
concentran aproximadamente el 60% de la cartera total, patrón que se ha 
mantenido estable durante todo el período analizado.

### Hallazgo 4 — Sectores de mayor exposición
El sector **{sector_lider}** lidera en volumen de créditos empresariales, 
seguido de Industria Manufacturera y Construcción. Estos tres sectores 
concentran más del 50% de la cartera empresarial del sistema.

### Nota metodológica
La SBS modificó su clasificación de sectores económicos entre 2020 y 2022, 
por lo que algunos sectores presentan cambios de nomenclatura en la serie histórica 
(ej. "Hoteles y Restaurantes" → "Alojamiento y Servicios de Comidas").
Los datos de 2020 incluyen el efecto de las medidas de reprogramación 
crediticia por COVID-19, lo que artificialmente redujo la morosidad reportada.
"""

display(Markdown(conclusiones))

In [ ]:
# Verificar contra valores conocidos
# BCP morosidad total dic 2024 debería estar ~4-5% según reportes públicos
print("=== VERIFICACIÓN DE MOROSIDAD ===\n")

print("Morosidad por banco - Diciembre 2024:")
mora_verificacion = (
    morosidad[morosidad['periodo'] == '2024-12']
    .groupby('banco')['morosidad_pct']
    .mean()
    .sort_values(ascending=False)
    .round(2)
)
print(mora_verificacion.to_string())

print("\nMorosidad BCP por tipo de crédito - Diciembre 2024:")
print(
    morosidad[
        (morosidad['periodo'] == '2024-12') &
        (morosidad['banco'] == 'BCP')
    ][['tipo_credito', 'morosidad_pct']]
    .sort_values('morosidad_pct', ascending=False)
    .to_string()
)

In [ ]:
RUTA_EXPORT = r'C:\Users\sebga\OneDrive\Documentos\Proyectos\Analisis-SBS\data\processed'
import os

# Exportar versiones limpias
morosidad.to_csv(os.path.join(RUTA_EXPORT, 'morosidad_limpia.csv'), index=False)
creditos_tipo.to_csv(os.path.join(RUTA_EXPORT, 'creditos_tipo_limpio.csv'), index=False)
creditos_sector.to_csv(os.path.join(RUTA_EXPORT, 'creditos_sector_limpio.csv'), index=False)

print("✔ morosidad_limpia.csv exportado")
print("✔ creditos_tipo_limpio.csv exportado")
print("✔ creditos_sector_limpio.csv exportado")
print(f"\nRegistros exportados:")
print(f"  Morosidad    : {len(morosidad):,}")
print(f"  Créditos tipo: {len(creditos_tipo):,}")
print(f"  Créditos sector: {len(creditos_sector):,}")